In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")

checks = {
    "no duplicate order lines":      f"SELECT COUNT(*) FROM (SELECT order_line_id FROM {catalog}.silver.orders GROUP BY 1 HAVING COUNT(*) > 1)",
    "no bad quantities":             f"SELECT COUNT(*) FROM {catalog}.silver.orders WHERE quantity <= 0",
    "statuses are lower case":       f"SELECT COUNT(*) FROM {catalog}.silver.orders WHERE status <> LOWER(status)",
    "no cancelled orders in silver": f"SELECT COUNT(*) FROM {catalog}.silver.orders WHERE status = 'cancelled'",
    "one current row per customer":  f"SELECT COUNT(*) FROM (SELECT customer_id FROM {catalog}.silver.customers WHERE is_current GROUP BY 1 HAVING COUNT(*) > 1)",
    "no duplicate live events":      f"SELECT COUNT(*) - COUNT(DISTINCT order_id, customer_id, product_id, order_ts) FROM {catalog}.silver.order_events",
}

failed = []
for name, query in checks.items():
    n = spark.sql(query).first()[0]
    print("PASS" if n == 0 else "FAIL", name, n)
    if n:
        failed.append(name)

if failed:
    raise Exception(f"Quality checks failed: {failed}")
print("All silver quality checks passed")